# DEDALUS · Perception · Demo su Kaggle
**YOLO trova veicoli e pedoni nelle foto registrate in CARLA, e la profondità dà la loro distanza.**
È lo stesso codice di `02_riconosci.py`, diviso in celle e con le immagini mostrate qui sotto.

### Prima di iniziare (una volta sola)
1. Account Kaggle con il **telefono verificato** (*Settings → Phone verification*), altrimenti GPU e Internet non si attivano.
2. A destra, in **Settings**: *Accelerator* → **GPU T4 x2** (o P100), *Internet* → **On**.
3. **Add Input** → scegli il dataset con la cartella `dati/` registrata da Alessandro.

### Poi
Menu **Run All**. Se cambi solo una cosa, cambiala nella cella **2 · Impostazioni** e rilancia da lì in giù.

## 1 · Installazione e controllo della GPU

In [ ]:
!pip install -q ultralytics

import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ATTENZIONE: nessuna GPU. Settings -> Accelerator -> GPU T4 (serve il telefono verificato)")

## 2 · Impostazioni
La cella cerca da sola le sessioni registrate dentro `/kaggle/input`. Se ne trova più di una, sceglie quella con *test* nel nome.
Per usarne un'altra, scrivi a mano il percorso in `CARTELLA` (lo trovi nel pannello *Input* a destra).

In [ ]:
import glob, os

BASE_INPUT = "/kaggle/input"
BASE_OUTPUT = "/kaggle/working"

sessioni = sorted(os.path.dirname(p) for p in glob.glob(os.path.join(BASE_INPUT, "**", "rgb"), recursive=True))
print("Sessioni trovate:")
for s in sessioni:
    print("  ", s)
if not sessioni:
    raise SystemExit("Nessuna cartella 'rgb' trovata: hai aggiunto il dataset con Add Input?")

test = [s for s in sessioni if "test" in os.path.basename(s)]
CARTELLA = test[0] if test else sessioni[0]       # <-- oppure scrivi qui il percorso a mano
CARTELLA_RISULTATI = os.path.join(BASE_OUTPUT, "risultati_" + os.path.basename(CARTELLA))

MODELLO = "yolo26n.pt"           # n = nano (il piu' veloce). Prova anche yolo26s.pt
CONFIDENZA_MINIMA = 0.3          # sotto questo valore ignoriamo il riquadro
FOV = 90.0                       # deve essere lo stesso di 01_registra.py
LARGHEZZA_CORSIA = 3.5           # metri: per dire se un ostacolo e' "davanti a noi"

CLASSI_UTILI = {"person": "pedone", "bicycle": "veicolo", "car": "veicolo",
                "motorcycle": "veicolo", "bus": "veicolo", "truck": "veicolo"}
COLORI = {"veicolo": (0, 165, 255), "pedone": (255, 0, 255)}   # arancione, magenta (B, G, R)

print("\nUso la sessione:", CARTELLA)
print("Risultati in:   ", CARTELLA_RISULTATI)

## 3 · Le due funzioni (uguali a `02_riconosci.py`)
- `distanza_del_riquadro`: mediana della profondità nel **centro** del riquadro (ai bordi c'è lo sfondo).
- `spostamento_laterale`: modello **pinhole**, `X = (u − cx) · Z / f`, cioè metri a destra (+) o a sinistra (−).

In [ ]:
import math
import numpy as np

def distanza_del_riquadro(profondita, x1, y1, x2, y2):
    larghezza, altezza = x2 - x1, y2 - y1
    cx1 = int(x1 + 0.3 * larghezza)
    cx2 = int(x2 - 0.3 * larghezza)
    cy1 = int(y1 + 0.3 * altezza)
    cy2 = int(y2 - 0.3 * altezza)
    pezzo = profondita[cy1:cy2 + 1, cx1:cx2 + 1]
    if pezzo.size == 0:
        return None
    return float(np.median(pezzo))


def spostamento_laterale(x_pixel, distanza, larghezza_immagine):
    f = larghezza_immagine / (2 * math.tan(math.radians(FOV) / 2))
    cx = larghezza_immagine / 2
    return (x_pixel - cx) * distanza / f

## 4 · YOLO su tutte le foto della sessione

In [ ]:
import time
import cv2
from ultralytics import YOLO

modello = YOLO(MODELLO)          # la prima volta scarica il modello da internet
os.makedirs(CARTELLA_RISULTATI, exist_ok=True)

nomi = sorted(f[:-4] for f in os.listdir(os.path.join(CARTELLA, "rgb")) if f.endswith(".png"))
print("Immagini da analizzare:", len(nomi))

tempi = []
righe = ["foto,classe,confidenza,distanza_m,laterale_m,davanti_a_noi"]
oggetti_per_foto = {}
video = None

for nome in nomi:
    img = cv2.imread(os.path.join(CARTELLA, "rgb", nome + ".png"))
    profondita = np.load(os.path.join(CARTELLA, "profondita", nome + ".npy"))

    inizio = time.time()
    risultato = modello(img, conf=CONFIDENZA_MINIMA, verbose=False)[0]
    ms = (time.time() - inizio) * 1000
    tempi.append(ms)

    trovati = []
    for riquadro in risultato.boxes:
        nome_yolo = modello.names[int(riquadro.cls)]
        if nome_yolo not in CLASSI_UTILI:
            continue
        classe = CLASSI_UTILI[nome_yolo]
        confidenza = float(riquadro.conf)
        x1, y1, x2, y2 = [int(v) for v in riquadro.xyxy[0].tolist()]

        distanza = distanza_del_riquadro(profondita, x1, y1, x2, y2)
        if distanza is None:
            testo = f"{nome_yolo} {confidenza:.2f}"
            righe.append(f"{nome},{classe},{confidenza:.2f},,,")
        else:
            laterale = spostamento_laterale((x1 + x2) / 2, distanza, img.shape[1])
            davanti = abs(laterale) < LARGHEZZA_CORSIA / 2
            testo = f"{nome_yolo} {confidenza:.2f} | {distanza:.1f} m"
            if davanti:
                testo += " | DAVANTI"
            righe.append(f"{nome},{classe},{confidenza:.2f},{distanza:.2f},{laterale:.2f},{davanti}")
            trovati.append((classe, distanza))

        colore = COLORI[classe]
        cv2.rectangle(img, (x1, y1), (x2, y2), colore, 2)
        cv2.putText(img, testo, (x1, max(y1 - 6, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, colore, 2)

    oggetti_per_foto[nome] = trovati
    cv2.putText(img, f"{nome}  |  {ms:.0f} ms", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.imwrite(os.path.join(CARTELLA_RISULTATI, nome + ".jpg"), img)
    if video is None:
        video = cv2.VideoWriter(os.path.join(CARTELLA_RISULTATI, "video.mp4"),
                                cv2.VideoWriter_fourcc(*"mp4v"), 4, (img.shape[1], img.shape[0]))
    video.write(img)

video.release()
with open(os.path.join(CARTELLA_RISULTATI, "rilevamenti.csv"), "w") as f:
    f.write("\n".join(righe))
print("Fatto!")

## 5 · Velocità (per la slide 9)

In [ ]:
tempi_utili = tempi[1:] if len(tempi) > 1 else tempi     # la prima foto e' lenta (YOLO si "scalda")
media = sum(tempi_utili) / len(tempi_utili)
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Tempo medio: {media:.1f} ms per immagine  ->  {1000 / media:.1f} immagini al secondo")
print(f"Hardware: Kaggle, {gpu}")
print()
print("Frase per la slide 9:")
print(f"  Speed: {media:.0f} ms per image ({1000 / media:.0f} FPS) on Kaggle {gpu}")

## 6 · Cosa ha trovato
Una tabella con tutti gli oggetti, e qualche numero riassuntivo.

In [ ]:
import pandas as pd
tabella = pd.read_csv(os.path.join(CARTELLA_RISULTATI, "rilevamenti.csv"), dtype={"foto": str})
print("Oggetti trovati in totale:", len(tabella))
print(tabella["classe"].value_counts().to_string())
print("Oggetti 'davanti a noi':", int((tabella["davanti_a_noi"] == True).sum()))
tabella.head(15)

## 7 · Immagini da guardare (per scegliere gli esempi della slide 9)
- **A**: le foto con **più oggetti** trovati: buoni candidati per "dove funziona".
- **B**: foto prese a caso, dove cercare cosa **manca** (pedoni lontani, auto coperte…). Guardatele bene: YOLO non segna ciò che non trova!

In [ ]:
import matplotlib.pyplot as plt
import random

def mostra(nomi_scelti, titolo):
    n = len(nomi_scelti)
    if n == 0:
        print("niente da mostrare")
        return
    colonne = 2
    righe_fig = (n + colonne - 1) // colonne
    plt.figure(figsize=(16, 4.6 * righe_fig))
    for i, nome in enumerate(nomi_scelti):
        img = cv2.imread(os.path.join(CARTELLA_RISULTATI, nome + ".jpg"))
        plt.subplot(righe_fig, colonne, i + 1)
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(nome)
        plt.axis("off")
    plt.suptitle(titolo, fontsize=16)
    plt.tight_layout()
    plt.show()

piu_oggetti = sorted(oggetti_per_foto, key=lambda k: len(oggetti_per_foto[k]), reverse=True)[:4]
mostra(piu_oggetti, "A · Foto con piu' oggetti trovati")

random.seed(0)
a_caso = random.sample(nomi, min(4, len(nomi)))
mostra(a_caso, "B · Foto a caso: cosa manca?")

## 8 · Scaricare i risultati
Dopo questa cella, a destra nel pannello **Output** trovi `risultati.zip`: dentro ci sono tutte le immagini con i riquadri, `video.mp4` e `rilevamenti.csv`.

In [ ]:
import shutil
archivio = shutil.make_archive(os.path.join(BASE_OUTPUT, "risultati"), "zip", CARTELLA_RISULTATI)
print("Pronto da scaricare:", archivio)

## Per la slide 9
1. Scegli **un'immagine dove funziona** e **una dove sbaglia** (dalla cella 7 o dallo zip), e trascinale nella slide al posto dei riquadri tratteggiati.
2. Sotto ciascuna, una frase: es. *"Car found at 12 m, ahead of us"*, *"Far pedestrian missed"*.
3. In fondo, la frase della **cella 5** con millisecondi, FPS e GPU.